[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/reto_expres/Reto_expres_1_series.ipynb)

# Reto exprés S12-13 · Series temporales: ¿qué ocupación habrá mañana?

**Solo se tocan las celdas ✏️** (cambiar un valor y pulsar ▶). Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · probar cuántos días mira la red hacia atrás y compararla con los métodos simples, **mirando solo la validación**
- ✏️ **2** · contadlo: titular con cifra y gráfico
- ✏️ **3** · la aplicación: predecir «mañana»
- Al final sale una **FICHA** con todos los números para vuestra presentación.

| Qué se predice | Datos | Rival simple | Métrica |
|---|---|---|---|
| ocupación de mañana | 1 096 días (`ocupacion_diaria.csv`) | «igual que ayer» y «igual que hace 7 días» | error medio en puntos (menos = mejor) |

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, re, urllib.request, time
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"
def dato(nombre):
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta): return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEMILLA = 42

from tensorflow import keras
keras.utils.set_random_seed(SEMILLA)
print("Listo ✔")

### ⚙️ Preparar la serie — pulsa ▶

**Ojo:** en una serie **no se baraja**: entrenamiento = el pasado, validación = lo siguiente, prueba = lo último. La red mira los últimos V días y predice el siguiente.

In [ ]:
d = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"])
fechas, serie, UNIDAD = d["fecha"], d["ocupacion_pct"].to_numpy(float), "puntos de ocupación"
MAXV = 28                                   # todas las pruebas empiezan en el mismo día para compararse bien
idx = np.arange(MAXV, len(serie))           # días que se pueden predecir
n = len(idx); i1, i2 = int(n * 0.6), int(n * 0.8)
idx_ent, idx_val, idx_pru = idx[:i1], idx[i1:i2], idx[i2:]
mu, sd = serie[:idx_ent[-1]].mean(), serie[:idx_ent[-1]].std()     # solo con el pasado de entrenamiento
z = (serie - mu) / sd

def ventanas(V, ids): return np.stack([z[t - V:t] for t in ids]).astype("float32"), z[ids].astype("float32")
mae = lambda real, pred: float(np.mean(np.abs(np.asarray(real) - np.asarray(pred))))
real = lambda ids: serie[ids]
BASE_AYER = lambda ids: serie[ids - 1]; BASE_SEM = lambda ids: serie[ids - 7]
print(f"{len(serie)} días · entrenamiento {len(idx_ent)} · validación {len(idx_val)} · prueba {len(idx_pru)} (en orden temporal)")
print(f"Rival 1 «igual que ayer»: error medio en validación {mae(real(idx_val), BASE_AYER(idx_val)):.2f} {UNIDAD}")
print(f"Rival 2 «igual que hace 7 días»: error medio en validación {mae(real(idx_val), BASE_SEM(idx_val)):.2f} {UNIDAD}")
plt.figure(figsize=(11, 2.8)); plt.plot(fechas, serie, lw=.7); plt.title("La serie completa"); plt.ylabel("% de ocupación"); plt.show()

### ⚙️ La red — pulsa ▶

In [ ]:
def construir(V, neuronas=32):
    m = keras.Sequential([keras.layers.Input(shape=(V,)), keras.layers.Dense(neuronas, activation="relu"), keras.layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mae"); return m

def entrenar(V, neuronas=32, epocas=150):
    keras.utils.set_random_seed(SEMILLA)
    Xe, ye = ventanas(V, idx_ent); Xv, yv = ventanas(V, idx_val)
    m = construir(V, neuronas)
    parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
    h = m.fit(Xe, ye, validation_data=(Xv, yv), epochs=epocas, batch_size=32, callbacks=[parar], verbose=0)
    return m, h

def predecir(m, V, ids): return m.predict(ventanas(V, ids)[0], verbose=0).ravel() * sd + mu
print("Listo ✔")

### ✏️ 1 · Probar cuántos días mira la red y elegir (solo con la validación)

Cambiad **VENTANAS** (días hacia atrás; mínimo 7, máximo 28). Mirad **val_error** (**más bajo = mejor**) y comparadlo con las dos filas de los rivales simples. Escribid en **ELEGIDA** la ventana que justificáis.

In [ ]:
VENTANAS = [7, 14, 28]
filas = [dict(candidata="igual que ayer", val_error=round(mae(real(idx_val), BASE_AYER(idx_val)), 2), epocas="-", segundos="-"),
         dict(candidata="igual que hace 7 días", val_error=round(mae(real(idx_val), BASE_SEM(idx_val)), 2), epocas="-", segundos="-")]
for V in VENTANAS:
    t0 = time.time(); m, h = entrenar(V)
    filas.append(dict(candidata=f"red mirando {V} días", val_error=round(mae(real(idx_val), predecir(m, V, idx_val)), 2),
                      epocas=len(h.history["loss"]), segundos=round(time.time() - t0)))
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = 14               # ✏️ días que mira la red, justificados con la tabla de arriba

### ⚙️ La prueba, una sola vez — pulsa ▶

La red solo vale la pena si **supera al mejor rival simple**. Si no, también es un resultado: explicad por qué.

In [ ]:
modelo, hist = entrenar(ELEGIDA)
pred = predecir(modelo, ELEGIDA, idx_pru); rr = real(idx_pru)
e_red, e_ayer, e_sem = mae(rr, pred), mae(rr, BASE_AYER(idx_pru)), mae(rr, BASE_SEM(idx_pru))
mejor_rival = min(e_ayer, e_sem); mejora = (mejor_rival - e_red) / mejor_rival
peores = np.argsort(-np.abs(rr - pred))[:5]
print("Los 5 días con más error:")
for j in peores: print(f"  {fechas.iloc[idx_pru[j]].date()} ({fechas.iloc[idx_pru[j]].day_name()}): real {rr[j]:.1f} · red {pred[j]:.1f} · error {abs(rr[j] - pred[j]):.1f}")
FICHA = dict(elegida=ELEGIDA, ayer=round(e_ayer, 2), semana=round(e_sem, 2), red=round(e_red, 2), mejora=round(mejora, 3))
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ predecir la ocupación de mañana · {len(serie)} días
Candidatas ...... {[(f['candidata'], f['val_error']) for f in filas]}  (error en validación)
Elegida ......... red mirando {ELEGIDA} días
--- PRUEBA (una sola vez; error medio en puntos de ocupación) ---
«Igual que ayer» .......... {e_ayer:.2f}
«Igual que hace 7 días» ... {e_sem:.2f}
Red ....................... {e_red:.2f}
Mejora sobre el mejor rival simple: {mejora:.1%} {"(la red gana)" if mejora > 0.03 else "(la red NO gana de forma clara)"}
============================================================
""")

### ✏️ 2 · Contadlo (S13): titular con cifra y gráfico

Cambiad **TITULAR** por vuestra conclusión (con cifras de variables). El gráfico de la derecha enseña **qué día de la semana falla más** (barras desde 0).
**Cómo se cuenta (S13):** el titular es una **conclusión con cifra** (no «Resultados»), el gráfico empieza en **0** y las cifras salen de variables (f-string), no se teclean a mano. Terminad con un **límite**: ¿cuándo no os fiaríais?

In [ ]:
TITULAR = f"La red se equivoca {e_red:.1f} puntos de ocupación de media; repetir la semana pasada, {e_sem:.1f}"   # ✏️ vuestra conclusión
dia = fechas.iloc[idx_pru].dt.dayofweek.to_numpy()
por_dia = pd.Series(np.abs(rr - pred)).groupby(dia).mean().reindex(range(7))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6)); ult = slice(-60, None)
ax[0].plot(fechas.iloc[idx_pru][ult], rr[ult], lw=2, label="real"); ax[0].plot(fechas.iloc[idx_pru][ult], pred[ult], label="red")
ax[0].plot(fechas.iloc[idx_pru][ult], BASE_SEM(idx_pru)[ult], ls=":", label="hace 7 días"); ax[0].legend(); ax[0].set_title("Últimos 60 días de la prueba"); ax[0].set_ylabel("% de ocupación")
ax[1].bar(["L", "M", "X", "J", "V", "S", "D"], por_dia.values, color="tab:blue"); ax[1].set_ylim(0, None); ax[1].set_title("Error medio por día de la semana"); ax[1].set_ylabel("puntos de ocupación")
fig.suptitle(TITULAR, fontsize=12, fontweight="bold"); plt.tight_layout(); plt.show()
FICHA["titular"] = TITULAR; FICHA["peor_dia"] = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"][int(por_dia.idxmax())]

### ✏️ 3 · La aplicación: predecir «mañana»

`mañana()` usa los últimos días reales de la serie y predice el día siguiente al último dato.

In [ ]:
def mañana():
    ultimo = fechas.iloc[-1]; ventana = z[-ELEGIDA:].reshape(1, -1).astype("float32")
    pr = float(modelo.predict(ventana, verbose=0)[0, 0] * sd + mu)
    print(f"Último dato: {ultimo.date()} · predicción para {(ultimo + pd.Timedelta(days=1)).date()}: {pr:.1f}% de ocupación")
    print(f"(para comparar, «igual que hace 7 días» diría {serie[-7]:.1f} y «igual que ayer» {serie[-1]:.1f})")

mañana()

### 🎤 Para la presentación (2–3 min)

Copiad la **FICHA**, la tabla de la celda ✏️ 1 y el gráfico. Una frase para cada pregunta: **¿gana a los métodos simples? · ¿por qué esa ventana? · ¿qué día falla más? · ¿lo usaríamos de verdad?**